In [1]:
import sys
sys.path.insert(0, '..')

from src.pipeline import SRRagPipeline
from src.generation.prompt_manager import build_prompt, prompt_stats, should_refuse, SYSTEM_PROMPTS
from src.indexing.embeddings import embed_query
from src.indexing.vector_store import load_vector_store, query_collection
import pandas as pd

pipeline = SRRagPipeline(persist_dir='../vector_store', collection_name='sr_papers', llm_provider='mock')
collection = load_vector_store(persist_dir='../vector_store', collection_name='sr_papers')
print(f'Collection: {collection.count()} chunks')

/Users/madhumithakatam/Documents/Projects/SR_RAG/sr-rag/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Collection: 695 chunks


In [2]:
question = "What loss function does SRGAN use?"
qv = embed_query(question)
chunks = query_collection(collection, qv, top_k=5)

print("=== Template comparison ===\n")
for template in ['v1', 'v2', 'v3_concise']:
    stats = prompt_stats(question, chunks, template=template)
    print(f"Template : {template}")
    print(f"  System chars : {stats['system_chars']}")
    print(f"  User chars   : {stats['user_chars']}")
    print(f"  Total chars  : {stats['total_chars']}")
    print(f"  Chunks used  : {stats['chunks_used']} / {stats['chunks_in']}")
    print()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10605.46it/s]


=== Template comparison ===

Template : v1
  System chars : 511
  User chars   : 4738
  Total chars  : 5249
  Chunks used  : 5 / 5

Template : v2
  System chars : 691
  User chars   : 4738
  Total chars  : 5429
  Chunks used  : 5 / 5

Template : v3_concise
  System chars : 160
  User chars   : 4738
  Total chars  : 4898
  Chunks used  : 5 / 5



In [3]:
# Artificially lower the budget to see truncation in action
from src.generation.prompt_manager import build_context

context, included = build_context(chunks, max_chars=1500)
print(f"Chunks in    : {len(chunks)}")
print(f"Chunks used  : {len(included)}")
print(f"Context len  : {len(context)} chars")
print()
print("--- Context preview ---")
print(context[:600])

Context budget hit (1500 chars). Included 2 / 5 chunks.


Chunks in    : 5
Chunks used  : 2
Context len  : 1524 chars

--- Context preview ---
[1] SRGAN / SRResNet (2016) — srgan_2016.pdf, page 7
Table 1: Performance of different loss functions for SR-
ResNet and the adversarial networks on Set5 and Set14
benchmark data. MOS score significantly higher (p < 0.05)
than with other losses in that category∗. [4× upscaling]
SRResNet-
SRGAN-
Set5
MSE
VGG22
MSE
VGG22
VGG54
PSNR
32.05
30.51
30.64
29.84
29.40
SSIM
0.9019
0.8803
0.8701
0.8468
0.8472
MOS
3.37
3.46
3.77
3.78
3.58
Set14
PSNR
28.49
27.19
26.92
26.44
26.02
SSIM
0.8184
0.7807
0.7611
0.7518
0.7397
MOS
2.98
3.15∗
3.43
3.57
3.72∗
• SRGAN-MSE: lSR
MSE, to investigate the adversarial
netw


In [4]:
# Ask something completely off-topic — scores will be low
qv_weak = embed_query("What is the capital of France?")
weak_chunks = query_collection(collection, qv_weak, top_k=5)

print("Weak query scores:")
for c in weak_chunks:
    print(f"  {c['method']} | score={c['score']:.3f}")

refuse, reason = should_refuse(weak_chunks)
print(f"\nRefuse: {refuse} | Reason: {reason}")

Weak query scores:
  SwinIR | score=0.197
  RDN | score=0.168
  VDSR | score=0.150
  HAT | score=0.143
  HAT | score=0.131

Refuse: True | Reason: low_score:0.197<0.3


In [5]:
result = pipeline.query("What is the capital of France?")
print(f"Refused: {result.get('refused', False)}")
print(f"Answer : {result['answer']}")

Refused: False
Answer : The provided corpus does not contain enough information to answer this question confidently. Please rephrase your question or check whether the relevant paper is included in the corpus.


In [6]:
questions = [
    "What loss function does SRGAN use?",
    "How does SwinIR use transformer attention?",
    "What is PSNR?",
    "Compare EDSR and SRResNet.",
    "Which papers use DIV2K?",
    "What is the capital of France?",   # should refuse
    "Who wrote Hamlet?",                 # should refuse
    "What datasets does RCAN use?",
]

rows = []
for q in questions:
    qv = embed_query(q)
    ch = query_collection(collection, qv, top_k=5)
    stats = prompt_stats(q, ch)
    rows.append({
        'Question':     q[:45],
        'Top score':    round(stats['top_score'], 3),
        'Refused':      stats['refused'],
        'Chunks used':  stats['chunks_used'],
        'Total chars':  stats['total_chars'],
    })

pd.DataFrame(rows)

,Question,Top score,Refused,Chunks used,Total chars
0,What loss function does SRGAN use?,0.688,False,5,5429
1,How does SwinIR use transformer attention?,0.630,False,5,4729
2,What is PSNR?,0.537,False,5,4664
3,Compare EDSR and SRResNet.,0.535,False,5,5323
4,Which papers use DIV2K?,0.392,False,5,5367
5,What is the capital of France?,0.197,True,0,876
6,Who wrote Hamlet?,0.076,True,0,876
7,What datasets does RCAN use?,0.521,False,5,5227


In [7]:
for name, prompt in SYSTEM_PROMPTS.items():
    print(f"=== {name} ({len(prompt)} chars) ===")
    print(prompt)
    print()

=== v1 (511 chars) ===
You are an expert assistant on image super-resolution (SR) research.
You answer questions strictly based on the provided paper excerpts.

Rules:
- Ground every claim in the provided context. Cite sources as [1], [2], etc.
- If the context does not contain enough information, say so clearly.
- Do not hallucinate methods, numbers, or paper names not present in the context.
- Be concise and precise. Use technical language appropriate for ML researchers.
- When comparing methods, structure your answer clearly.

=== v2 (691 chars) ===
You are a knowledgeable research assistant specialising in image super-resolution.
Your answers are grounded exclusively in the paper excerpts provided below.

Guidelines:
- Every factual claim must cite its source using [N] notation.
- If multiple papers support a claim, cite all relevant ones: [1][3].
- If the provided context is insufficient to answer confidently, respond with:
  "The provided corpus does not contain enough informatio